# 4 · Retrieval-augmented generation

⏱ about 11 minutes

Now the two halves meet: **retrieve** the best chunks, **augment** the prompt with them, and let
the model **generate** an answer from that text.

![The prompt: rules, the 5 numbered chunks and the question go to the LLM](img/rag-prompt.svg)

First the same setup as in notebook 3: load the index into Qdrant and define `search()`.

In [ ]:
import os

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv()
CHAT_MODEL = os.environ["CHAT_MODEL"]
EMBED_MODEL = os.environ["EMBED_MODEL"]
llm = OpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])
import json

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

chunks = json.load(open("index.json"))
db = QdrantClient(":memory:")
db.create_collection(
    "papers",
    vectors_config=VectorParams(size=len(chunks[0]["vector"]), distance=Distance.COSINE),
)
db.upsert(
    "papers",
    points=[
        PointStruct(id=i, vector=c["vector"], payload={k: c[k] for k in ("source", "page", "text")})
        for i, c in enumerate(chunks)
    ],
)


def search(question, limit=5):
    vector = llm.embeddings.create(model=EMBED_MODEL, input=[question]).data[0].embedding
    return db.query_points("papers", query=vector, limit=limit).points

The question from notebook 1, this time with context. Compare the answer with what you got there.

In [ ]:
QUESTION = "In Kage et al. (2018), what were the lifetime-encoded beads loaded with, and which lifetimes did they have?"

hits = search(QUESTION)
context = "\n\n".join(f"[{i}] {hit.payload['text']}" for i, hit in enumerate(hits, start=1))

SYSTEM_PROMPT = (
    "Answer the question using only the context below. "
    "If the context does not contain the answer, say that you don't know. "
    "After each statement, cite the context it comes from as [1], [2], …"
)

reply = llm.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {QUESTION}"},
    ],
)
answer = reply.choices[0].message.content
display(Markdown(answer))

## Citations

The chunks in the context are numbered `[1]` to `[5]`, and the prompt asks the model to cite those
numbers. A regular expression finds them in the answer, and each number leads back to a search hit,
which knows its file and page.

![The [1] in the answer leads to the first search hit, whose payload names the file and page](img/citations.svg)

In [ ]:
import re

for number in sorted({int(n) for n in re.findall(r"\[(\d+)\]", answer)}):
    if 1 <= number <= len(hits):
        print(f"[{number}] = {hits[number - 1].payload['source']}, page {hits[number - 1].payload['page']}")

## Is the answer any good?

So far you judged answers by eye. **Faithfulness** turns that into a number: a second model
call, the *judge*, splits the answer into claims and checks each one against the chunks.

`faithfulness = supported claims ÷ all claims`

It needs no hand-written correct answer, only the answer and the chunks it was built from.

In [ ]:
JUDGE = (
    "Split the answer into its individual factual claims. For each claim, decide whether the context supports it. "
    'Reply with JSON only, in this form: {"claims": [{"claim": "...", "supported": true}]}'
)


def faithfulness(answer, context):
    reply = llm.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": JUDGE},
            {"role": "user", "content": f"Context:\n{context}\n\nAnswer:\n{answer}"},
        ],
        response_format={"type": "json_object"},
    )
    claims = json.loads(reply.choices[0].message.content)["claims"]
    for c in claims:
        print("✓" if c["supported"] else "✗", c["claim"])
    if not claims:  # "I don't know" makes no claims, so there is nothing to check
        return None
    return round(sum(c["supported"] for c in claims) / len(claims), 2)


print("faithfulness:", faithfulness(answer, context))

Now the kind of answer you got in notebook 1, without the paper, judged against the same chunks:

In [ ]:
no_context = llm.chat.completions.create(model=CHAT_MODEL, messages=[{"role": "user", "content": QUESTION}])
print("faithfulness:", faithfulness(no_context.choices[0].message.content, context))

Look at the ✗ lines. That is what you couldn't see in notebook 1. Would you still rely on it?

> The judge is a language model too, so treat the score as a strong hint, not a verdict. The
> full template computes faithfulness and a second score, *relevance* (does the answer address the
> question?), for every answer, and lets you compare configurations.

## The answering bot in Chainlit

Everything from this notebook in one app:
- the search runs inside a `cl.Step`, which you can unfold in the chat to see what was found,
- only the question and the answer go into the history; the context is fetched fresh every time,
- every cited `[n]` becomes a `cl.Pdf` link that opens the paper at the right page.

In [ ]:
%%writefile app.py
import json
import os
import re

import chainlit as cl
from dotenv import load_dotenv
from openai import AsyncOpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

load_dotenv()
llm = AsyncOpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])
# Bonus: what happens without "say that you don't know"?
SYSTEM_PROMPT = (
    "Answer the question using only the context below. "
    "If the context does not contain the answer, say that you don't know. "
    "After each statement, cite the context it comes from as [1], [2], …"
)

chunks = json.load(open("index.json"))
db = QdrantClient(":memory:")
db.create_collection(
    "papers",
    vectors_config=VectorParams(size=len(chunks[0]["vector"]), distance=Distance.COSINE),
)
db.upsert(
    "papers",
    points=[
        PointStruct(id=i, vector=c["vector"], payload={k: c[k] for k in ("source", "page", "text")})
        for i, c in enumerate(chunks)
    ],
)


async def search(question, limit=5):
    response = await llm.embeddings.create(model=os.environ["EMBED_MODEL"], input=[question])
    return db.query_points("papers", query=response.data[0].embedding, limit=limit).points


@cl.on_chat_start
async def on_chat_start():
    cl.user_session.set("history", [{"role": "system", "content": SYSTEM_PROMPT}])


@cl.on_message
async def on_message(message: cl.Message):
    async with cl.Step(name="Search the papers", type="retrieval") as step:
        hits = await search(message.content)
        # Bonus: show the text of every chunk, not only file and page.
        step.output = "\n\n".join(
            f"**{h.payload['source']}, p. {h.payload['page']}** ({h.score:.2f})" for h in hits
        )

    context = "\n\n".join(f"[{i}] {h.payload['text']}" for i, h in enumerate(hits, start=1))
    history = cl.user_session.get("history")
    prompt = {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {message.content}"}

    answer = cl.Message(content="")
    stream = await llm.chat.completions.create(
        model=os.environ["CHAT_MODEL"], messages=history + [prompt], stream=True
    )
    async for part in stream:
        if part.choices and part.choices[0].delta.content:
            await answer.stream_token(part.choices[0].delta.content)

    # Turn every cited [n] into a link that opens the PDF at the right page.
    pdfs = {}
    for number in sorted({int(n) for n in re.findall(r"\[(\d+)\]", answer.content)}):
        if 1 <= number <= len(hits):
            source, page = hits[number - 1].payload["source"], hits[number - 1].payload["page"]
            name = f"{source.removesuffix('.pdf')}, p. {page}"
            pdfs[name] = cl.Pdf(name=name, path=f"data/{source}", page=page, display="side")
    if pdfs:
        answer.content += "\n\n**Sources:** " + " · ".join(pdfs)
        answer.elements = list(pdfs.values())
    await answer.send()

    history.append({"role": "user", "content": message.content})
    history.append({"role": "assistant", "content": answer.content})

In [ ]:
import socket
import subprocess
import sys

# Start the chatbot, unless it's still running from an earlier notebook.
with socket.socket() as probe:
    running = probe.connect_ex(("127.0.0.1", 8000)) == 0
if not running:
    subprocess.Popen(
        [sys.executable, "-m", "chainlit", "run", "app.py", "--watch", "--headless"],
        stdout=open("chainlit.log", "w"),
        stderr=subprocess.STDOUT,
    )
print("Chatbot: http://localhost:8000")

## Your turn

Ask the bot the Kage question in the browser, click a source, and unfold the search step.

1. **How much context?** In the cell that asks `QUESTION` with context, call
   `search(QUESTION, limit=2)`, run that cell and the faithfulness cell, then do the same with
   `limit=15`. Which answer is more complete? Which score is higher? A faithful answer can still
   leave things out: the score checks what was said, not what is missing. Put `%%time` as the
   first line of that cell to see what 15 chunks cost in time.

**Bonus: strict or not?** In the `app.py` cell, remove *"If the context does not contain the
answer, say that you don't know."* and ask the bot *What is the capital of France?* Then put it
back. Which behaviour do you want for your own documents, and why?

**Bonus: show the evidence.** Make the search step show the text of every chunk, not only file
and page, so you can check the answer against it yourself.

**Bonus: follow-up questions.** Ask *What were the lifetime-encoded beads in Kage et al. loaded
with?*, then *And which lifetimes did they have?* The second search only sees the second question,
which doesn't say what "they" are. Fix it: before searching, let the LLM rewrite the question so
it makes sense without the conversation, and search with that.

<details>
<summary><b>Solutions</b> (try first!)</summary>

**1** — `%%time` as the first line of the cell with the context, and one changed line in it:
```python
hits = search(QUESTION, limit=2)   # then limit=15
```
In testing, `limit=2` gave a shorter answer without the lifetimes, and still scored 1.0: everything
it said was backed, it just left the lifetimes out.

**Strict or not** — delete the sentence from `SYSTEM_PROMPT`, run the cell, start a new chat.

**Show the evidence**
```python
        step.output = "\n\n".join(
            f"**[{i}] {h.payload['source']}, p. {h.payload['page']}** ({h.score:.2f})\n\n{h.payload['text']}"
            for i, h in enumerate(hits, start=1)
        )
```

**Follow-up questions** — a new function, and use it at the start of `on_message`:
```python
async def standalone(question, history):
    if len(history) < 2:  # only the system prompt: nothing to refer back to
        return question
    conversation = "\n".join(f"{m['role']}: {m['content']}" for m in history[1:])
    response = await llm.chat.completions.create(
        model=os.environ["CHAT_MODEL"],
        messages=[{"role": "user", "content": (
            f"Conversation so far:\n{conversation}\n\n"
            "Rewrite this follow-up question so it can be understood without the conversation. "
            f"Reply with the question only.\n\nFollow-up: {question}"
        )}],
    )
    return response.choices[0].message.content


@cl.on_message
async def on_message(message: cl.Message):
    query = await standalone(message.content, cl.user_session.get("history"))
    async with cl.Step(name=f"Search: {query}", type="retrieval") as step:
        hits = await search(query)
        ...
```
</details>

## Questions people ask

**Why did the answer without the paper not score 0?** Part of it can be right: the beads did contain
quantum dots. The judge tells you which part, claim by claim.

**The judge is the same model as the one answering. Is that a problem?** It can be. Treat the score as
a strong hint, not a verdict, and use a different judge model when it matters.

**Why not put all the PDFs into the prompt?** Models with long context windows can take a lot of
text, but every question would then pay for every page, in time and cost. Page citations only work
if you mark every page yourself. And a collection of 200 papers is larger than any context window.
Retrieval sends only the five chunks that matter, and each one knows its file and page.